In [ ]:
import os
from pathlib import Path
import pandas as pd
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
# from sklearn.impute import SimpleImputer

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import optuna

import mlflow
from mlflow.pyfunc.model import PythonModel
from mlflow.data.pandas_dataset import from_pandas

import joblib

In [30]:
ROOT_DIR = Path.cwd().parent
PROCESSED_DATA_DIR = ROOT_DIR / "data/processed"
TRAINED_MODEL_PATH = ROOT_DIR / "models/trained/random_forest.pkl"
LABEL_ENCODER_PATH = ROOT_DIR / "models/trained/label_encoder.pkl"
TEMP_DIR = ROOT_DIR / "tmp"

STUDY_NAME = "random_forest"
N_TRIALS = 200

MLFLOW_TRACKING_URI = "http://localhost:5000"       # url to self hosted mlflow.
MLFLOW_EXPERIMENT_NAME = "student_performance"
MLFLOW_RUN_NAME = "random_forest"

Load processed dataset.

In [31]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / 'X_train.csv')
X_test = pd.read_csv(PROCESSED_DATA_DIR / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DATA_DIR / 'y_train.csv').squeeze('columns')
y_test = pd.read_csv(PROCESSED_DATA_DIR / 'y_test.csv').squeeze('columns')

Full pipeline.

In [32]:
nominal_cols = [
    'school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 
    'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic'
]

nominal_transformer = Pipeline(steps=[
    # ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='error', sparse_output=False, drop="first"))
])

categorical_transformer = ColumnTransformer(
    transformers=[
        # ('ordinal', ordinal_transformer, ordinal_cols),
        ('nominal', nominal_transformer, nominal_cols)
    ],
    remainder='passthrough'     # for numerical cols
)

estimator = Pipeline(steps=[
    ('preprocessing', categorical_transformer),
    ('classifier', RandomForestClassifier())
])

estimator

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('nominal', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If 

In [33]:
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

Hyperparams search w Optuna.

In [34]:
def objective(trial):
    constant_params = {
        'random_state': 42,
        'n_jobs': -1,
        'class_weight': None
    }

    search_spaces = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7, None]),
        'max_samples': trial.suggest_float('max_samples', 0.5, 1.0, step=0.1),
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'ccp_alpha': trial.suggest_float('ccp_alpha', 1e-4, 0.05, log=True)
    }

    trial.set_user_attr("constant_params", constant_params)

    estimator_clone = clone(estimator)
    estimator_clone.named_steps['classifier'].set_params(**constant_params, **search_spaces)

    scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    score = cross_validate(
        X=X_train,
        y=y_train_encoded,
        estimator=estimator_clone,
        scoring=scoring,
        n_jobs=1,
        cv=cv,
        return_train_score=True
    )

    test_accuracy = score['test_accuracy'].mean()
    test_precision_macro = score['test_precision_macro'].mean()
    test_recall_macro = score['test_recall_macro'].mean()
    test_f1_macro = score['test_f1_macro'].mean()

    return test_accuracy, test_precision_macro, test_recall_macro, test_f1_macro

study = optuna.create_study(
    storage="sqlite:///db.sqlite3",
    study_name=STUDY_NAME,
    directions=['maximize', 'maximize', 'maximize', 'maximize'],
    load_if_exists=True,
)

n_jobs = os.cpu_count() // 2  # type: ignore

study.optimize(
    func=objective,
    n_trials=N_TRIALS,
    n_jobs=n_jobs
)

[I 2026-08-01 11:11:48,218] Using an existing study with name 'random_forest' instead of creating a new one.


[I 2026-08-01 11:11:52,176] Trial 600 finished with values: [0.4928547690262546, 0.5069713708024601, 0.49284165176334627, 0.49081604640124343] and parameters: {'n_estimators': 50, 'max_depth': 13, 'min_samples_split': 18, 'min_samples_leaf': 6, 'max_features': 0.7, 'max_samples': 0.5, 'criterion': 'gini', 'ccp_alpha': 0.00023633277658350067}.
[I 2026-08-01 11:11:53,898] Trial 601 finished with values: [0.4618525376252196, 0.49586493706728757, 0.45969730716200546, 0.45326628449018846] and parameters: {'n_estimators': 150, 'max_depth': 13, 'min_samples_split': 18, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_samples': 0.6, 'criterion': 'gini', 'ccp_alpha': 0.026885350994739547}.
[I 2026-08-01 11:11:55,109] Trial 602 finished with values: [0.4646773963822817, 0.4793205175911619, 0.46505317651017264, 0.4643723778492665] and parameters: {'n_estimators': 300, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': 'log2', 'max_samples': 0.8, 'criterion': 'gini',

In [35]:
!optuna-dashboard sqlite:///db.sqlite3

[2026-08-01 11:21:01 +0700] [159592] [INFO] Starting gunicorn 26.0.0
[2026-08-01 11:21:01 +0700] [159592] [INFO] Listening at: http://127.0.0.1:8080 (159592)
[2026-08-01 11:21:01 +0700] [159592] [INFO] Using worker: gthread
[2026-08-01 11:21:01 +0700] [159643] [INFO] Booting worker with pid: 159643
[2026-08-01 11:21:01 +0700] [159592] [INFO] Control socket listening at /run/user/1000/gunicorn.ctl
/home/ansha/projects/end-to-end-mlops-pipeline-student-performance-classification/.venv/lib/python3.12/site-packages/optuna_dashboard/_importance.py:71: ExperimentalWarning: PedAnovaImportanceEvaluator is experimental (supported from v3.6.0). The interface can change in the future.
  study, target=lambda t: t.values[objective_id], evaluator=PedAnovaImportanceEvaluator()
/home/ansha/projects/end-to-end-mlops-pipeline-student-performance-classification/.venv/lib/python3.12/site-packages/optuna_dashboard/_importance.py:70: UserWarning: PedAnovaImportanceEvaluator computes the importances of param

Best trial 793 w test_accuracy, test_precision_macro, test_recall_macro, test_f1_macro: 
0.5098039215686274, 0.5213886965904319, 0.510538659030315, 0.5091222397639742.

In [36]:
study = optuna.load_study(
    storage="sqlite:///db.sqlite3",
    study_name=STUDY_NAME,
)

selected_trial = study.trials[793]

print(selected_trial.user_attrs)
print(selected_trial.params)

{'constant_params': {'random_state': 42, 'n_jobs': -1, 'class_weight': None}}
{'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 5, 'max_features': 0.5, 'max_samples': 0.8, 'criterion': 'entropy', 'ccp_alpha': 0.0007207957306333559}


Log to MLflow

In [44]:
mlflow.set_tracking_uri(uri=MLFLOW_TRACKING_URI)
mlflow.set_experiment(experiment_name=MLFLOW_EXPERIMENT_NAME)

class ModelWrapper(PythonModel):
    def __init__(self):
        self.label_encoder = None
        self.trained_model = None

    def load_context(self, context):
        import joblib

        self.label_encoder = joblib.load(context.artifacts["label_encoder_path"])
        self.trained_model = joblib.load(context.artifacts["trained_model_path"])

    def predict(self, context, model_input, params=None):
        numeric_predictions = self.trained_model.predict(model_input)

        return self.label_encoder.inverse_transform(numeric_predictions)


timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
with mlflow.start_run(run_name=f"{MLFLOW_RUN_NAME}_{timestamp}"):
    dataset = from_pandas(df=X_train, name="student_performance")
    mlflow.log_input(dataset, context="train")

    params = {**selected_trial.user_attrs["constant_params"], **selected_trial.params}
    mlflow.log_params(params)

    estimator.named_steps['classifier'].set_params(**selected_trial.user_attrs["constant_params"], **selected_trial.params)
    estimator.fit(X_train, y_train_encoded)

    # temporary save
    joblib.dump(estimator, TRAINED_MODEL_PATH)
    joblib.dump(label_encoder, LABEL_ENCODER_PATH)

    mlflow.pyfunc.log_model(
        artifact_path="random_forest",
        artifacts={
            "label_encoder_path": str(LABEL_ENCODER_PATH),
            "trained_model_path": str(TRAINED_MODEL_PATH),
        },
        python_model=ModelWrapper(),
    )

    y_test_encoded = label_encoder.transform(y_test)
    y_pred = estimator.predict(X_test)

    report = classification_report(y_test_encoded, y_pred, output_dict=True)
    formatted_report = {f"{k}_{sk}": sv for k, v in report.items() if isinstance(v, dict) for sk, sv in v.items()}
    mlflow.log_metrics(formatted_report)

2026/08/01 12:04:16 INFO mlflow.tracking.fluent: Experiment with name 'student_performance' does not exist. Creating a new experiment.
/home/ansha/projects/end-to-end-mlops-pipeline-student-performance-classification/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/home/ansha/projects/end-to-end-mlops-pipeline-student-performance-classification/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to in

🏃 View run random_forest_20260801-120416 at: http://localhost:5000/#/experiments/1/runs/1ca498ee4e264cb28f72c7459d8b6c67
🧪 View experiment at: http://localhost:5000/#/experiments/1
